# Agent Memory

The LangGraph agents from [notebook 08](/courses/llm-eng/08-agents-langgraph.html) carry state inside the `TypedDict` that flows through the graph — a message list that grows with each turn and is discarded when the session ends. This works for single-session tasks but fails for a financial analyst assistant that needs to remember that a client prefers aggressive growth portfolios, that last quarter's portfolio review flagged a concentration risk in tech equities, and that the client's compliance threshold is Basel III Tier 1. These facts live across sessions, not within one. This deep dive covers the full memory taxonomy: short-term memory that manages the in-context budget, long-term episodic memory that retrieves relevant past sessions, and semantic memory that distills and persists extracted facts. We conclude by assembling all three into a `FinancialAnalystAgent` and running a three-session demo.

Setup:

In [ ]:
#| echo: false
import os, json
import numpy as np
from dotenv import load_dotenv
load_dotenv()

import openai
from pydantic import BaseModel
from typing import Optional, Type

PRICES = {
    "gpt-4o":      {"input": 2.50,  "output": 10.00},
    "gpt-4o-mini": {"input": 0.15,  "output": 0.60},
}

class LLMClient:
    def __init__(self, model="gpt-4o-mini", temperature=0.0):
        self.model = model; self.temperature = temperature
        self._client = openai.OpenAI()
        self._in = 0; self._out = 0

    def complete(self, messages, *, response_format=None):
        if response_format is not None:
            resp = self._client.beta.chat.completions.parse(
                model=self.model, messages=messages,
                temperature=self.temperature, response_format=response_format)
        else:
            resp = self._client.chat.completions.create(
                model=self.model, messages=messages, temperature=self.temperature)
        if resp.usage:
            self._in += resp.usage.prompt_tokens; self._out += resp.usage.completion_tokens
        if response_format is not None: return resp.choices[0].message.parsed
        return resp.choices[0].message.content

    @property
    def total_cost(self):
        if self.model not in PRICES: return 0.0
        p = PRICES[self.model]
        return (self._in * p["input"] + self._out * p["output"]) / 1_000_000

llm = LLMClient()
llm_strong = LLMClient(model="gpt-4o")

## Memory Taxonomy

Agent memory splits along two axes: **duration** (how long it persists) and **storage** (where it lives). The following table fixes the vocabulary we use throughout this notebook:

| Type | Subtype | Duration | Storage | Purpose |
|---|---|---|---|---|
| **Short-term** | Conversation buffer | One session | In-context | Full turn history in the prompt |
| **Short-term** | Sliding window | One session | In-context | Last $k$ turns; oldest dropped |
| **Short-term** | Summarization | One session | In-context + LLM | Compress old turns to a running summary |
| **Long-term** | Episodic | Cross-session | Vector store | Retrieve past session summaries by semantic similarity |
| **Long-term** | Semantic | Cross-session | Vector store | Distilled facts extracted from past sessions |
| **External** | Tool-backed | Persistent | DB / API | Real-time data; the agent calls a retrieval tool |

<br>

**Context budget.** Let $C$ denote the context window size in tokens (e.g. $C = 128{,}000$ for `gpt-4o`). At inference time, the prompt occupies a budget $B \leq C$ distributed across system instructions, retrieved memory, and the current conversation. Short-term memory strategies differ in how they spend this budget:

- **Buffer:** all $n$ turns are included. Token usage grows as $O(n)$, with no bound until $B = C$.
- **Sliding window:** only the last $k$ turns are included. Token usage is $O(k)$, bounded by construction.
- **Summarization:** old turns are compressed to a summary of length $s \ll \sum_{i=1}^{n-k} |\text{turn}_i|$. Token usage is $O(s + k)$ where $s$ is the summary budget and $k$ is the recent-turn window.

Long-term memory adds a retrieval step before assembling the prompt: the agent embeds the current query, searches the vector store, and injects the top-$m$ results. The cost is $O(s_\text{ep} \cdot m)$ where $s_\text{ep}$ is the average length of an episode summary.

:::{.callout-note}
The context window $C$ is a hard constraint enforced by the model API — exceeding it raises an error. In production, short-term memory management is not optional: even `gpt-4o`'s 128k window fills in roughly 200 turns of a typical analyst conversation.

:::

## Short-Term: Buffer and Sliding Window

We implement `BufferMemory` and `SlidingWindowMemory` as classes that manage the message list passed to the LLM. Both expose the same interface: `.add(role, content)` to append a turn, `.messages()` to retrieve what should go into the prompt, and `.token_count()` to report the current budget usage.

In [ ]:
import tiktoken

_enc = tiktoken.encoding_for_model("gpt-4o-mini")


def count_tokens(text: str) -> int:
    """Count tokens in a string using the gpt-4o-mini tokenizer."""
    return len(_enc.encode(text))


class BufferMemory:
    """Stores every turn in the conversation unconditionally."""

    def __init__(self, system_prompt: str = ""):
        self._system = system_prompt
        self._turns: list[dict] = []

    def add(self, role: str, content: str) -> None:
        self._turns.append({"role": role, "content": content})

    def messages(self) -> list[dict]:
        """Return full prompt: system + all turns."""
        prefix = [{"role": "system", "content": self._system}] if self._system else []
        return prefix + self._turns

    def token_count(self) -> int:
        return sum(count_tokens(m["content"]) for m in self.messages())


class SlidingWindowMemory:
    """Retains only the last k turns; older turns are dropped."""

    def __init__(self, k: int = 10, system_prompt: str = ""):  # <1>
        self._k = k
        self._system = system_prompt
        self._turns: list[dict] = []

    def add(self, role: str, content: str) -> None:
        self._turns.append({"role": role, "content": content})
        if len(self._turns) > self._k:  # <2>
            self._turns = self._turns[-self._k :]

    def messages(self) -> list[dict]:
        prefix = [{"role": "system", "content": self._system}] if self._system else []
        return prefix + self._turns

    def token_count(self) -> int:
        return sum(count_tokens(m["content"]) for m in self.messages())

1. The window parameter $k$ counts individual messages (alternating user/assistant), not turn pairs. With $k = 10$ we retain 5 full exchanges.
2. After appending the new turn, we truncate `_turns` to the last $k$ entries. This is an O(1) slice — Python list slicing copies the reference list, not the underlying strings.

We simulate a 20-turn analyst session and compare token growth for both memory types:

In [ ]:
SYSTEM = "You are a financial analyst assistant. Answer questions about portfolio risk concisely."

# Synthetic analyst session turns (user, assistant pairs)
ANALYST_TURNS = [
    ("user",      "What is our current CET1 ratio?"),
    ("assistant", "The CET1 ratio is 14.8%, well above the 4.5% regulatory minimum and our 13% internal target."),
    ("user",      "How does that compare to Q3?"),
    ("assistant", "In Q3 the CET1 ratio was 14.3%. The 50bps improvement reflects retained earnings and modest RWA reduction."),
    ("user",      "What is our one-day 99th percentile VaR?"),
    ("assistant", "The 99th percentile one-day VaR is $142M across the consolidated trading book."),
    ("user",      "How is that split across asset classes?"),
    ("assistant", "Equities: $58M, fixed income: $47M, FX: $24M, commodities: $13M. Correlations reduce the sum to $142M."),
    ("user",      "What is the LCR?"),
    ("assistant", "The LCR is 128%, meaning $280B in HQLA covers at least 100% of 30-day net cash outflows under stress."),
    ("user",      "Any margin call exposure this week?"),
    ("assistant", "Three accounts triggered margin calls totaling $18M. All were resolved within the 24-hour cure period."),
    ("user",      "Is the tech equity concentration risk from last quarter resolved?"),
    ("assistant", "Yes. We reduced the FAANG position from 22% to 15% of the growth portfolio, within the 20% single-sector cap."),
    ("user",      "What is our Basel III leverage ratio?"),
    ("assistant", "The Basel III leverage ratio is 5.8%, above the 3% minimum. Total exposure measure is $1.6T."),
    ("user",      "Net interest margin trend?"),
    ("assistant", "NIM expanded 18bps to 2.94% this quarter, driven by higher short-term rates partially offset by deposit repricing."),
    ("user",      "What about credit loss provisions?"),
    ("assistant", "Provision for credit losses increased to $2.1B, reflecting normalization from the historically low post-COVID levels."),
    ("user",      "Investment banking revenue outlook?"),
    ("assistant", "IB revenues declined 23% to $6.1B. Improving capital markets conditions suggest a recovery in H2 2025."),
    ("user",      "Summarize the three biggest risks heading into next quarter."),
    ("assistant", "(1) Rate sensitivity: NIM expansion slows if the Fed pauses. (2) Credit normalization: provisions may increase further. (3) IB recovery risk: M&A pipeline is fragile if deal conditions deteriorate."),
]

buf_mem  = BufferMemory(system_prompt=SYSTEM)
win_mem  = SlidingWindowMemory(k=8, system_prompt=SYSTEM)

buf_tokens  = []
win_tokens  = []

for role, content in ANALYST_TURNS:
    buf_mem.add(role, content)
    win_mem.add(role, content)
    buf_tokens.append(buf_mem.token_count())
    win_tokens.append(win_mem.token_count())

print(f"After {len(ANALYST_TURNS)} messages:")
print(f"  BufferMemory:         {buf_tokens[-1]:,} tokens")
print(f"  SlidingWindowMemory:  {win_tokens[-1]:,} tokens (k=8)")

We plot the token count growth for both strategies across the 20-turn session:

In [ ]:
#| code-fold: true
%config InlineBackend.figure_formats = ['svg']
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

turns = list(range(1, len(ANALYST_TURNS) + 1))

fig, ax = plt.subplots(figsize=(7, 3.5))
ax.plot(turns, buf_tokens, label="Buffer (unbounded)",       color="#c0392b", linewidth=2)
ax.plot(turns, win_tokens, label="Sliding window ($k=8$)",   color="#2980b9", linewidth=2)
ax.set_xlabel("Turn", fontsize=11)
ax.set_ylabel("Tokens in prompt", fontsize=11)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{int(x):,}"))
ax.grid(alpha=0.4, linestyle="dashed")
ax.legend(fontsize=10)
ax.set_title("Token budget: buffer vs. sliding window", fontsize=12)
fig.tight_layout()
plt.show()

**Figure.** Buffer memory grows linearly with each turn — the analyst session above reaches roughly 700 tokens after 20 messages. Sliding window plateaus once the buffer fills the $k=8$ window, holding steady regardless of session length. The plateau level depends only on $k$ and the average message length.

## Short-Term: Summarization Memory

Sliding window drops old turns entirely — the agent genuinely cannot remember what was said at turn 1 by turn 20. Summarization memory avoids this: when the buffer exceeds a token threshold $T$, the oldest turns are compressed by the LLM into a running summary. The prompt then contains the summary (compact) plus the $k$ most recent turns (full fidelity). Formally, if we denote the summary at step $n$ as $\sigma_n$ and the recent window as $\mathcal{W}_k$, the context sent to the model is:

$$\text{prompt} = [\text{system}] \;\|\; [\sigma_n] \;\|\; \mathcal{W}_k$$

where $\|$ denotes concatenation. When a new summary is generated, the compression ratio $r = |\sigma_n| \,/\, \sum_{i=1}^{n-k} |\text{turn}_i|$ measures how much information was compressed; values of $r \approx 0.1$–$0.2$ are typical.

In [ ]:
class SummarizationMemory:
    """Compresses old turns into a running summary when the buffer exceeds T tokens."""

    def __init__(self, llm_client: LLMClient, system_prompt: str = "",
                 token_threshold: int = 400, recent_k: int = 6):  # <1>
        self._llm = llm_client
        self._system = system_prompt
        self._threshold = token_threshold
        self._k = recent_k
        self._summary: str = ""
        self._turns: list[dict] = []
        self.compression_log: list[dict] = []

    def add(self, role: str, content: str) -> None:
        self._turns.append({"role": role, "content": content})
        if self.token_count() > self._threshold:  # <2>
            self._compress()

    def _compress(self) -> None:
        """Summarize turns older than the recent window and update running summary."""
        to_summarize = self._turns[: -self._k]  # <3>
        self._turns   = self._turns[-self._k :]

        old_tokens = sum(count_tokens(t["content"]) for t in to_summarize)
        old_tokens += count_tokens(self._summary)

        history_text = "\n".join(
            f"{t['role'].upper()}: {t['content']}" for t in to_summarize
        )
        prompt_content = (
            f"Previous summary:\n{self._summary}\n\n"
            f"New conversation:\n{history_text}\n\n"
            "Write an updated concise summary (3–5 sentences) of the full conversation "
            "so far, focusing on key financial figures, risks identified, and decisions made."
        )
        self._summary = self._llm.complete([
            {"role": "user", "content": prompt_content}
        ])

        new_tokens = count_tokens(self._summary)
        self.compression_log.append({  # <4>
            "old_tokens": old_tokens,
            "new_tokens": new_tokens,
            "ratio": new_tokens / max(old_tokens, 1),
        })

    def messages(self) -> list[dict]:
        """Return [system, summary-as-user-note, recent turns]."""
        msgs = []
        if self._system:
            msgs.append({"role": "system", "content": self._system})
        if self._summary:
            msgs.append({"role": "system",
                         "content": f"[Conversation summary so far]:\n{self._summary}"})  # <5>
        msgs.extend(self._turns)
        return msgs

    def token_count(self) -> int:
        return sum(count_tokens(m["content"]) for m in self.messages())

1. `token_threshold=400` is deliberately low so the compression fires during the short demo session. In production, a typical threshold is 70–80% of the context window.
2. The check fires after every `add` call. If the total prompt already exceeds the threshold after appending the newest turn, we compress immediately before the next LLM call.
3. We split `_turns` into the chunk to summarize (all but the last $k$) and the recent window to keep verbatim. The split point is `–k` which is a standard Python negative index.
4. We log the compression event: how many tokens were consumed by the turns-to-summarize, how many the new summary uses, and the compression ratio $r = \text{new} / \text{old}$.
5. We inject the running summary as a second `system` message. This ensures it is never confused with a user turn by the model — the OpenAI API allows multiple `system` messages.

We replay the analyst session through `SummarizationMemory` and inspect the compression log:

In [ ]:
sum_mem = SummarizationMemory(
    llm_client=llm,
    system_prompt=SYSTEM,
    token_threshold=400,
    recent_k=6,
)
sum_tokens = []

for role, content in ANALYST_TURNS:
    sum_mem.add(role, content)
    sum_tokens.append(sum_mem.token_count())

print(f"Final token count: {sum_tokens[-1]:,} (summarization) vs {buf_tokens[-1]:,} (buffer)")
print(f"Compression events: {len(sum_mem.compression_log)}")
for i, ev in enumerate(sum_mem.compression_log, 1):
    print(f"  Event {i}: {ev['old_tokens']} → {ev['new_tokens']} tokens  (ratio={ev['ratio']:.2f})")

print(f"\nFinal summary:\n{sum_mem._summary}")

## Long-Term: Episodic Memory

Episodic memory stores summaries of past sessions in a vector store. When a new session begins, the agent embeds the opening query and retrieves the top-$m$ most semantically similar past episodes. These are injected into the system prompt as context, giving the agent access to information that would otherwise be lost between sessions.

We use the same numpy vector store pattern as [notebook 03](/courses/llm-eng/03-rag-concepts.html) — a simple in-process store with cosine similarity search. Each episode is stored with its embedding, its summary text, and a timestamp used later for recency scoring.

In [ ]:
import time


def embed(texts: list[str]) -> np.ndarray:
    """Embed a list of strings with text-embedding-3-small. Returns (n, 1536) float32."""
    client = openai.OpenAI()
    resp = client.embeddings.create(model="text-embedding-3-small", input=texts)
    return np.array([e.embedding for e in resp.data], dtype=np.float32)


class EpisodicMemory:
    """Vector store of past session summaries, retrieved by semantic similarity."""

    def __init__(self):
        self._embeddings: np.ndarray | None = None   # (n, 1536)
        self._episodes: list[dict] = []              # {summary, timestamp, session_id}

    def store_episode(self, summary: str, session_id: str) -> None:  # <1>
        """Embed and store a session summary."""
        vec = embed([summary])  # (1, 1536)
        if self._embeddings is None:
            self._embeddings = vec
        else:
            self._embeddings = np.vstack([self._embeddings, vec])
        self._episodes.append({
            "summary": summary,
            "session_id": session_id,
            "timestamp": time.time(),
        })

    def retrieve(self, query: str, top_k: int = 3) -> list[dict]:  # <2>
        """Return top-k most relevant episodes by cosine similarity."""
        if self._embeddings is None or len(self._episodes) == 0:
            return []
        q = embed([query])  # (1, 1536)
        # Cosine similarity: dot product of unit vectors
        norms = np.linalg.norm(self._embeddings, axis=1, keepdims=True)
        normed = self._embeddings / np.clip(norms, 1e-8, None)
        q_norm = q / np.linalg.norm(q)
        scores = (normed @ q_norm.T).squeeze()  # (n,)
        if scores.ndim == 0:
            scores = scores[np.newaxis]
        top_idx = np.argsort(scores)[::-1][:top_k]
        return [
            {"score": float(scores[i]), **self._episodes[i]}
            for i in top_idx
        ]

    def __len__(self) -> int:
        return len(self._episodes)

1. `store_episode` appends the embedding to the matrix by calling `np.vstack`. For small episodic stores (hundreds of sessions) this is fine. For thousands of sessions, use a pre-allocated buffer or a persistent vector store.
2. `retrieve` normalizes all stored embeddings and the query to unit vectors before taking the dot product, which is equivalent to cosine similarity. The `np.clip` prevents division by zero for zero vectors.

We seed the episodic store with three past analyst sessions and then retrieve the most relevant one for a new query:

In [ ]:
PAST_SESSIONS = [
    {
        "session_id": "session-2024-10-15",
        "summary": (
            "Portfolio review for the Apex Growth Fund. The analyst flagged a 22% concentration "
            "in FAANG equities, exceeding the 20% single-sector cap. The client agreed to reduce "
            "the position to 15% over the following two weeks. CET1 ratio was 14.3%. "
            "Client preference: aggressive growth, willing to accept higher volatility."
        ),
    },
    {
        "session_id": "session-2024-11-02",
        "summary": (
            "Margin call review session. Three accounts triggered margin calls totaling $18M. "
            "All resolved within the 24-hour cure period. The client requested that margin call "
            "thresholds be reviewed quarterly. Compliance threshold: Basel III Tier 1, minimum 13% "
            "CET1 internal target."
        ),
    },
    {
        "session_id": "session-2024-12-01",
        "summary": (
            "Annual risk review. One-day 99th percentile VaR was $142M. NIM expanded 18bps to 2.94%. "
            "Provisions for credit losses increased to $2.1B — normalization trend expected to continue. "
            "Investment banking revenues declined 23% to $6.1B. Client asked for H2 2025 IB recovery "
            "scenario analysis."
        ),
    },
]

ep_mem = EpisodicMemory()
for sess in PAST_SESSIONS:
    ep_mem.store_episode(sess["summary"], sess["session_id"])
    print(f"Stored: {sess['session_id']}")

print(f"\nEpisodic store size: {len(ep_mem)} episodes")

Retrieving the most relevant past sessions for a new analyst query about portfolio concentration:

In [ ]:
query = "Has the tech equity concentration risk in the Apex Growth Fund been resolved?"
results = ep_mem.retrieve(query, top_k=3)

print(f"Query: {query}\n")
for r in results:
    print(f"[score={r['score']:.3f}] {r['session_id']}")
    print(f"  {r['summary'][:120]}...\n")

## Long-Term: Semantic Memory

Episodic memory retrieves full session summaries. Semantic memory goes one step further: it distills durable facts from those summaries and stores them individually. A fact like "Client prefers aggressive growth portfolios" is more useful as a standalone retrievable entry than buried inside a 200-token session summary.

The extraction step uses the LLM to identify atomic, reusable facts from a conversation. Each fact is embedded and stored. On each new turn, the top-$m$ most relevant facts are retrieved and injected into the system prompt — giving the agent persistent knowledge of client preferences, constraints, and history.

In [ ]:
class ExtractedFacts(BaseModel):
    facts: list[str]


class SemanticMemory:
    """Extracts and retrieves distilled facts from conversation sessions."""

    def __init__(self, llm_client: LLMClient):
        self._llm = llm_client
        self._embeddings: np.ndarray | None = None  # (n, 1536)
        self._facts: list[dict] = []                # {fact, source_session, timestamp}

    def extract_and_store(self, conversation_text: str, session_id: str) -> list[str]:  # <1>
        """Extract atomic facts from conversation text and store them."""
        messages = [
            {
                "role": "system",
                "content": (
                    "Extract 3–7 durable, atomic facts from this financial analyst conversation. "
                    "Focus on: client preferences, compliance thresholds, recurring risk flags, "
                    "agreed actions, and portfolio parameters. Each fact should be a single "
                    "self-contained sentence. Avoid facts that are transient (e.g. today's VaR)."
                ),
            },
            {"role": "user", "content": conversation_text},
        ]
        result: ExtractedFacts = self._llm.complete(messages, response_format=ExtractedFacts)
        vecs = embed(result.facts)  # (n_facts, 1536)
        if self._embeddings is None:
            self._embeddings = vecs
        else:
            self._embeddings = np.vstack([self._embeddings, vecs])
        for fact in result.facts:
            self._facts.append({
                "fact": fact,
                "source_session": session_id,
                "timestamp": time.time(),
            })
        return result.facts

    def retrieve(self, query: str, top_k: int = 5) -> list[dict]:  # <2>
        """Return top-k most relevant facts by cosine similarity."""
        if self._embeddings is None or len(self._facts) == 0:
            return []
        q = embed([query])
        norms = np.linalg.norm(self._embeddings, axis=1, keepdims=True)
        normed = self._embeddings / np.clip(norms, 1e-8, None)
        q_norm = q / np.linalg.norm(q)
        scores = (normed @ q_norm.T).squeeze()
        if scores.ndim == 0:
            scores = scores[np.newaxis]
        top_idx = np.argsort(scores)[::-1][:top_k]
        return [
            {"score": float(scores[i]), **self._facts[i]}
            for i in top_idx
        ]

    def __len__(self) -> int:
        return len(self._facts)

1. `extract_and_store` calls the LLM once per session to produce a structured `ExtractedFacts` list. We embed all facts in a single API call (batched input) to minimize latency.
2. The `retrieve` method is identical in structure to `EpisodicMemory.retrieve` — both use the same cosine similarity pattern. The difference is the granularity of what is stored: full summaries vs. individual atomic facts.

We extract facts from all three past sessions and inspect what was stored:

In [ ]:
sem_mem = SemanticMemory(llm_client=llm)

for sess in PAST_SESSIONS:
    facts = sem_mem.extract_and_store(sess["summary"], sess["session_id"])
    print(f"\nExtracted from {sess['session_id']} ({len(facts)} facts):")
    for f in facts:
        print(f"  · {f}")

print(f"\nTotal facts in semantic memory: {len(sem_mem)}")

Retrieving relevant facts for a new query about the client's risk preferences:

In [ ]:
query = "What are the client's risk preferences and compliance constraints?"
relevant_facts = sem_mem.retrieve(query, top_k=4)

print(f"Query: {query}\n")
for r in relevant_facts:
    print(f"[score={r['score']:.3f}] {r['fact']}")
    print(f"           ↳ from {r['source_session']}")

## Retrieval Scoring

Raw cosine similarity ranks memories by semantic relevance alone. The Generative Agents paper (Park et al., 2023) proposes a composite score that also accounts for recency and importance:

$$\text{score}(m, q) = \alpha \cdot \text{recency}(m) + \beta \cdot \text{importance}(m) + \gamma \cdot \text{relevance}(m, q)$$

where each component is independently normalized to $[0, 1]$ before weighting. We define:

- **Recency** — exponential decay from creation time: $\text{recency}(m) = \exp\!\left(-\lambda \cdot \Delta t\right)$ where $\Delta t$ is the age of the memory in hours and $\lambda$ controls the decay rate. With $\lambda = 0.01$, a memory loses half its recency weight in about 70 hours.

- **Importance** — a scalar scored 1–10 by the LLM at storage time, capturing how significant the memory is (a margin call event scores higher than a routine ratio check).

- **Relevance** — cosine similarity between the memory embedding and the query embedding, as computed above.

This prevents **memory staleness**: a highly relevant but very old memory competes against a moderately relevant but recent one. In financial services, stale memories are dangerous — a risk preference stated 18 months ago may no longer apply after a market regime change.

:::{.callout-tip}
Set $\alpha$ high for agents where recency matters most (live market commentary), $\beta$ high for agents where importance dominates (compliance incident tracking), and $\gamma$ high for pure Q&A workloads. A safe starting point for a financial analyst assistant is $\alpha = 0.3, \beta = 0.3, \gamma = 0.4$.

:::

In [ ]:
import math


class ImportanceScore(BaseModel):
    score: int   # 1–10
    reasoning: str


def score_importance(fact: str, llm_client: LLMClient) -> int:
    """Ask the LLM to rate the importance of a memory on a 1-10 scale."""
    messages = [
        {
            "role": "system",
            "content": (
                "Rate the importance of this financial analyst memory on a scale 1–10. "
                "10 = critical (compliance breach, margin call, concentrated risk). "
                "5 = significant (client preference, recurring metric). "
                "1 = routine (standard ratio well within limits)."
            ),
        },
        {"role": "user", "content": fact},
    ]
    result: ImportanceScore = llm_client.complete(messages, response_format=ImportanceScore)
    return result.score


def composite_score(
    relevance: float,
    importance: float,  # 1–10, will be normalized to 0–1
    timestamp: float,   # Unix timestamp
    alpha: float = 0.3,
    beta:  float = 0.3,
    gamma: float = 0.4,
    decay_rate: float = 0.01,  # λ per hour  # <1>
) -> float:
    """Compute the Generative Agents composite memory score."""
    age_hours = (time.time() - timestamp) / 3600.0
    recency = math.exp(-decay_rate * age_hours)           # <2>
    imp_norm = (importance - 1.0) / 9.0                   # <3>
    return alpha * recency + beta * imp_norm + gamma * relevance

1. `decay_rate=0.01` means a memory created 69 hours ago scores $e^{-0.69} \approx 0.5$ on recency — roughly half its original weight after about three days.
2. We compute age in hours and apply exponential decay. The recency score is always in $(0, 1]$ for non-negative ages.
3. Importance is rated 1–10 by the LLM. We normalize to $[0, 1]$ by mapping $[1, 10] \mapsto [0, 1]$ linearly: $\text{imp\_norm} = (\text{score} - 1) / 9.$

We demonstrate composite scoring on three toy memories with different age and importance profiles:

In [ ]:
now = time.time()
ONE_HOUR  = 3600
ONE_DAY   = 86400
ONE_WEEK  = 7 * ONE_DAY

toy_memories = [
    {
        "label":     "Recent margin call (1h ago)",
        "relevance": 0.82,
        "importance": 9,
        "timestamp": now - ONE_HOUR,
    },
    {
        "label":     "Old client preference (1 week ago)",
        "relevance": 0.91,
        "importance": 6,
        "timestamp": now - ONE_WEEK,
    },
    {
        "label":     "Routine CET1 check (1 day ago)",
        "relevance": 0.55,
        "importance": 3,
        "timestamp": now - ONE_DAY,
    },
]

print(f"{'Memory':<42} {'Recency':>8} {'Imp':>5} {'Relev':>7} {'Score':>7}")
print("-" * 74)
for m in toy_memories:
    age_h = (now - m["timestamp"]) / 3600
    recency = math.exp(-0.01 * age_h)
    sc = composite_score(m["relevance"], m["importance"], m["timestamp"])
    print(f"{m['label']:<42} {recency:>8.3f} {m['importance']:>5} {m['relevance']:>7.2f} {sc:>7.3f}")

## Full Agent Demo

We assemble a `FinancialAnalystAgent` that integrates all three memory types into a LangGraph-based agent. The architecture follows the pattern from [notebook 08](/courses/llm-eng/08-agents-langgraph.html): a `TypedDict` state flows through the graph, and the memory modules are managed as side-effects on node entry and exit.

**Session flow.** On session start, the agent (1) retrieves the top-3 relevant past episodes from `EpisodicMemory`, (2) retrieves the top-5 relevant facts from `SemanticMemory`, and injects both into the system prompt. During the session, `SummarizationMemory` manages the in-context budget. On session end, the agent stores a summary in `EpisodicMemory` and extracts new facts into `SemanticMemory`.

In [ ]:
from typing import TypedDict, Annotated
from langgraph.graph import StateGraph, END
from langgraph.graph.message import add_messages


class AnalystState(TypedDict):
    session_id: str
    messages: Annotated[list, add_messages]  # <1>
    episodic_context: str
    semantic_context: str
    user_input: str
    turn_count: int


class FinancialAnalystAgent:
    """LangGraph agent with short-term, episodic, and semantic memory."""

    BASE_SYSTEM = (
        "You are a senior financial analyst assistant. You provide precise, "
        "data-grounded answers about portfolio risk, capital ratios, and compliance. "
        "Always cite specific figures when available. Be concise."
    )

    def __init__(self, llm_client: LLMClient):
        self._llm = llm_client
        self.episodic  = EpisodicMemory()
        self.semantic  = SemanticMemory(llm_client)
        self._short_mem: SummarizationMemory | None = None
        self._graph = self._build_graph()

    def _build_graph(self):
        builder = StateGraph(AnalystState)
        builder.add_node("load_memory",  self._load_memory_node)
        builder.add_node("respond",      self._respond_node)
        builder.set_entry_point("load_memory")
        builder.add_edge("load_memory", "respond")
        builder.add_edge("respond", END)
        return builder.compile()

    def start_session(self, session_id: str) -> None:  # <2>
        """Initialize a new session with fresh short-term memory."""
        self._current_session_id = session_id
        system = self.BASE_SYSTEM
        self._short_mem = SummarizationMemory(
            llm_client=self._llm,
            system_prompt=system,
            token_threshold=600,
            recent_k=6,
        )

    def end_session(self) -> None:  # <3>
        """Store session summary and extract facts into long-term memory."""
        if self._short_mem is None:
            return
        history = "\n".join(
            f"{t['role'].upper()}: {t['content']}" for t in self._short_mem._turns
        )
        summary_prompt = (
            f"{self._short_mem._summary}\n\nRecent turns:\n{history}\n\n"
            "Write a 3–5 sentence session summary for long-term episodic memory, "
            "focusing on key decisions, risk figures, and client preferences expressed."
        )
        final_summary = self._llm.complete([
            {"role": "user", "content": summary_prompt}
        ])
        sid = self._current_session_id
        self.episodic.store_episode(final_summary, sid)
        self.semantic.extract_and_store(final_summary, sid)
        print(f"  [memory] Stored episode and extracted facts for {sid}.")

    def _load_memory_node(self, state: AnalystState) -> dict:
        """Retrieve episodic and semantic context relevant to the current user input."""
        query = state["user_input"]

        ep_results = self.episodic.retrieve(query, top_k=2)
        ep_context = "\n".join(
            f"- [{r['session_id']}] {r['summary']}" for r in ep_results
        ) if ep_results else "No prior sessions."

        sem_results = self.semantic.retrieve(query, top_k=4)
        sem_context = "\n".join(
            f"- {r['fact']}" for r in sem_results
        ) if sem_results else "No stored facts."

        return {"episodic_context": ep_context, "semantic_context": sem_context}

    def _respond_node(self, state: AnalystState) -> dict:
        """Generate a response using short-term memory + retrieved long-term context."""
        ep_ctx  = state["episodic_context"]
        sem_ctx = state["semantic_context"]

        # Inject long-term memory as a system message before the conversation  # <4>
        if ep_ctx or sem_ctx:
            context_block = (
                f"[Long-term memory: relevant past episodes]\n{ep_ctx}\n\n"
                f"[Long-term memory: relevant facts]\n{sem_ctx}"
            )
            self._short_mem.add("system", context_block)  # temporary injection

        self._short_mem.add("user", state["user_input"])
        response = self._llm.complete(self._short_mem.messages())
        self._short_mem.add("assistant", response)

        return {
            "messages": [{"role": "assistant", "content": response}],
            "turn_count": state["turn_count"] + 1,
        }

    def chat(self, user_input: str, session_id: str = "default") -> str:
        """Send a message and return the agent's response."""
        if self._short_mem is None:
            self.start_session(session_id)
        init_state: AnalystState = {
            "session_id": session_id,
            "messages": [],
            "episodic_context": "",
            "semantic_context": "",
            "user_input": user_input,
            "turn_count": 0,
        }
        result = self._graph.invoke(init_state)
        last = result["messages"][-1]
        return last.content if hasattr(last, "content") else last["content"]

1. `Annotated[list, add_messages]` accumulates assistant response messages across the graph invocation. This is the standard LangGraph pattern from notebook 08.
2. `start_session` resets `SummarizationMemory` to a clean state. Each session begins with an empty short-term buffer; long-term memory (episodic and semantic) is shared and persistent across sessions.
3. `end_session` triggers the long-term consolidation step: the LLM synthesizes a final session summary, which is stored in `EpisodicMemory`, and then `SemanticMemory.extract_and_store` distills atomic facts from it.
4. Long-term context is injected as a `system` role message appended to the short-term buffer just before the LLM call. This ensures the model treats it as background context rather than a user turn.

We initialize the agent with the three past sessions already in long-term memory and run a three-session demo:

In [ ]:
agent = FinancialAnalystAgent(llm_client=llm)

# Pre-populate long-term memory with the three past sessions
for sess in PAST_SESSIONS:
    agent.episodic.store_episode(sess["summary"], sess["session_id"])
    agent.semantic.extract_and_store(sess["summary"], sess["session_id"])

print(f"Long-term memory initialized: {len(agent.episodic)} episodes, {len(agent.semantic)} facts.")

**Session 1** — the analyst asks about portfolio concentration risk. The agent should recall the FAANG reduction agreement from the October session:

In [ ]:
agent.start_session("session-2025-01-10")

q1 = "Has the tech equity concentration issue in the Apex Growth Fund been resolved?"
r1 = agent.chat(q1, session_id="session-2025-01-10")
print(f"Q: {q1}")
print(f"A: {r1}\n")

q2 = "What is the client's stated preference on portfolio risk tolerance?"
r2 = agent.chat(q2, session_id="session-2025-01-10")
print(f"Q: {q2}")
print(f"A: {r2}\n")

agent.end_session()

**Session 2** — a new session asking about margin calls. The agent should recall both the November margin call resolution and the client's compliance threshold:

In [ ]:
agent.start_session("session-2025-01-17")

q3 = "Were there any margin calls last quarter, and what is our compliance threshold for CET1?"
r3 = agent.chat(q3, session_id="session-2025-01-17")
print(f"Q: {q3}")
print(f"A: {r3}\n")

q4 = "Did the client request any changes to the margin call process?"
r4 = agent.chat(q4, session_id="session-2025-01-17")
print(f"Q: {q4}")
print(f"A: {r4}\n")

agent.end_session()

**Session 3** — the agent should now have accumulated facts from both prior sessions stored in long-term memory. This query crosses information from multiple past sessions:

In [ ]:
agent.start_session("session-2025-01-24")

q5 = (
    "Given everything we know about this client's risk profile and compliance requirements, "
    "what are the three most important things to cover in today's quarterly review?"
)
r5 = agent.chat(q5, session_id="session-2025-01-24")
print(f"Q: {q5}")
print(f"A: {r5}\n")

print(f"\nLong-term memory at end of session 3:")
print(f"  Episodes: {len(agent.episodic)}")
print(f"  Facts:    {len(agent.semantic)}")
print(f"  LLM cost so far: ${agent._llm.total_cost:.4f}")

agent.end_session()

## Backend Tradeoffs

The `EpisodicMemory` and `SemanticMemory` implementations above use numpy arrays in-process. This is appropriate for a notebook demo but inadequate for a production analyst assistant serving multiple users. The table below summarizes when to use each backend:

| Backend | Latency | Persistence | Scalability | Cost | Use When |
|---|---|---|---|---|---|
| **In-process numpy** | < 1 ms | None (restarts wipe memory) | Single process | $0 | Prototyping, notebooks, single-user CLI tools |
| **Redis (with RediSearch)** | 1–5 ms | Yes (append-only log) | Horizontal (Redis Cluster) | Low ($0.05–$0.15/hr on AWS) | Low-latency read-heavy workloads, session caches, fact lookup at sub-5ms SLA |
| **pgvector (PostgreSQL)** | 5–20 ms | Yes (WAL, ACID) | Vertical + read replicas | Medium ($0.10–$0.50/hr on RDS) | Multi-user production, need SQL joins (e.g. user_id filter), audit trail requirements |
| **Managed (Pinecone, Weaviate)** | 5–50 ms | Yes | Fully managed | Higher ($0.50–$2.00/hr) | Large corpora (millions of episodes), no infra team |

<br>

**Regulatory note.** In financial services, long-term agent memory constitutes a processing activity under GDPR Article 5 — personal data must be accurate, limited to what is necessary, and not kept longer than needed. Episodic and semantic stores that contain client-attributable facts require (1) a defined retention policy, (2) a deletion mechanism, and (3) inclusion in the firm's data processing register. The `timestamp` field we store on every memory entry is the minimum metadata required to implement a retention policy.

:::{.callout-caution}
Do not store raw conversation transcripts in a vector store as episodic memories. Store summaries only. Raw transcripts may contain material non-public information (MNPI), PII, or confidential client data — all of which require stricter access controls and retention rules than a summary embedding.

:::

:::{.callout-note}
For pgvector, the minimum schema needed is: `(id UUID, user_id TEXT, content TEXT, embedding VECTOR(1536), importance FLOAT, created_at TIMESTAMPTZ)`. The `user_id` column enables row-level security policies so that one analyst's memories are never retrieved for another. The `importance` column supports the composite scoring formula without a separate LLM call at retrieval time — importance is scored once at storage time and stored alongside the embedding.

:::

## Appendix: Token Counting Across Memory Strategies

We compare the token profiles of all three short-term strategies on the 20-turn analyst session:

In [ ]:
#| code-fold: true
%config InlineBackend.figure_formats = ['svg']
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

turns = list(range(1, len(ANALYST_TURNS) + 1))

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(turns, buf_tokens,  label="Buffer (unbounded)",          color="#c0392b", linewidth=2)
ax.plot(turns, win_tokens,  label="Sliding window ($k=8$)",      color="#2980b9", linewidth=2)
ax.plot(turns, sum_tokens,  label="Summarization ($T=400, k=6$)", color="#27ae60", linewidth=2, linestyle="--")

for ev_idx, ev in enumerate(sum_mem.compression_log):
    ax.axvline(x=turns[ev_idx * (len(turns) // max(len(sum_mem.compression_log), 1))],
               color="#27ae60", alpha=0.3, linestyle=":")

ax.set_xlabel("Turn", fontsize=11)
ax.set_ylabel("Tokens in prompt", fontsize=11)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{int(x):,}"))
ax.grid(alpha=0.4, linestyle="dashed")
ax.legend(fontsize=10)
ax.set_title("Short-term memory strategy comparison", fontsize=12)
fig.tight_layout()
plt.show()

**Figure.** Summarization memory (green dashed) drops sharply after each compression event, then grows again as new turns accumulate. The result is bounded growth with better recall than sliding window — older turns are not discarded but compressed. Sliding window (blue) plateaus immediately at its steady-state level. Buffer memory (red) grows without bound.

---

$\blacksquare$